# Global Patent Intelligence Data Pipeline

Handles large `.tsv` files using chunking and efficient loading.

In [ ]:
import pandas as pd
import sqlite3
import json

# File paths (adjust if needed in Kaggle)
patents_path = 'g_patent\\g_patent.tsv'
abstracts_path = 'g_patent_abstract\\g_patent_abstract.tsv'
inventors_path = 'g_inventor_disambiguated\\g_inventor_disambiguated.tsv'
patent_inventor_path = 'g_persistent_inventor\\g_persistent_inventor.tsv'
companies_path = 'g_assignee_disambiguated\\g_assignee_disambiguated.tsv'
patent_company_path = 'g_persistent_assignee\\g_persistent_assignee.tsv'
locations_path = 'g_location_disambiguated\\g_location_disambiguated.tsv'

print('Paths set')

In [ ]:
def load_large_csv(path, cols, nrows=50000):
    return pd.concat(
        pd.read_csv(
            path,
            sep='\t',
            compression='infer',
            usecols=cols,
            chunksize=50000,
            nrows=nrows
        ),
        ignore_index=True
    )

print('loader loaded')

In [ ]:
# Load datasets
patents = load_large_csv(patents_path, ['patent_id','patent_title','patent_date'])

abstracts = load_large_csv(abstracts_path, ['patent_id','patent_abstract'])

inventors = load_large_csv(
    inventors_path,
    ['inventor_id','disambig_inventor_name_first','disambig_inventor_name_last','location_id']
)

patent_inventor = load_large_csv(
    patent_inventor_path,
    ['patent_id','inventor_sequence']
)

companies = load_large_csv(
    companies_path,
    ['assignee_id','disambig_assignee_organization']
)

patent_company = load_large_csv(
    patent_company_path,
    ['patent_id','disamb_assignee_id_20251231']
)

locations = load_large_csv(
    locations_path,
    ['location_id','disambig_country']
)

print('All data loaded')

In [ ]:
# Fix column names
locations.rename(columns={'disambig_country':'country'}, inplace=True)

patent_company.rename(
    columns={'disamb_assignee_id_20251231':'company_id'},
    inplace=True
)

companies.rename(
    columns={
        'assignee_id':'company_id',
        'disambig_assignee_organization':'name'
    },
    inplace=True
)

# ----------------------------
# Patent dataset
patents_clean = patents.merge(abstracts, on='patent_id', how='left')
patents_clean['year'] = pd.to_datetime(
    patents_clean['patent_date'], errors='coerce'
).dt.year

# ----------------------------
# Inventor dataset
inventors['name'] = (
    inventors['disambig_inventor_name_first'].fillna('') + ' ' +
    inventors['disambig_inventor_name_last'].fillna('')
)

inventors_clean = inventors.merge(
    locations,
    on='location_id',
    how='left'
)[['inventor_id','name','country']]

# ----------------------------
# Company dataset
companies_clean = companies[['company_id','name']].dropna()

# ----------------------------
# Relationships
relationships = patent_inventor.merge(
    patent_company,
    on='patent_id',
    how='left'
)

relationships.columns = ['patent_id','inventor_id','company_id']

print("Cleaning done")

In [ ]:
patents_clean.to_csv('working\\clean_patents.csv', index=False)
inventors_clean.to_csv('working\\clean_inventors.csv', index=False)
companies_clean.to_csv('working\\clean_companies.csv', index=False)
relationships.to_csv('working\\relationships.csv', index=False)

print('Saved cleaned files')

In [ ]:
import os
import sqlite3

os.makedirs('working', exist_ok=True)
conn = sqlite3.connect('working/patents.db')

print('db present')

patents_clean.to_sql(
    'patents', conn,
    if_exists='replace',
    index=False,
    chunksize=50000
)
print('patents saved')

inventors_clean.to_sql(
    'inventors', conn,
    if_exists='replace',
    index=False,
    chunksize=50000
)
print('inventors saved')
companies_clean.to_sql(
    'companies', conn,
    if_exists='replace',
    index=False,
    chunksize=50000
)
print('companies saved')

relationships.to_sql(
    'relationships', conn,
    if_exists='replace',
    index=False,
    chunksize=50000
)
print('relationships saved')

conn.close()

print("Database ready")

In [ ]:
import sqlite3
import pandas as pd

# OPEN NEW CONNECTION
conn = sqlite3.connect('working/patents.db')
print('Connected to database')

# Queries
top_inventors = pd.read_sql("""
SELECT i.name, COUNT(DISTINCT r.patent_id) as patents
FROM relationships r
JOIN inventors i ON r.inventor_id = i.inventor_id
GROUP BY i.name
ORDER BY patents DESC
LIMIT 10
""", conn)
print(top_inventors)

top_companies = pd.read_sql("""
SELECT c.name, COUNT(DISTINCT r.patent_id) as patents
FROM relationships r
JOIN companies c ON r.company_id = c.company_id
GROUP BY c.name
ORDER BY patents DESC
LIMIT 10
""", conn)
print(top_companies)

country_trends = pd.read_sql("""
SELECT country, COUNT(*) as total
FROM inventors
GROUP BY country
ORDER BY total DESC
""", conn)
print(country_trends)

year_trends = pd.read_sql("""
SELECT year, COUNT(*) as total
FROM patents
GROUP BY year
ORDER BY year
""", conn)
print(year_trends)

# CLOSE AFTER USE
conn.close()

print('Queries executed')

In [ ]:
# Save reports
top_inventors.to_csv('working\\top_inventors.csv', index=False)
top_companies.to_csv('working\\top_companies.csv', index=False)
country_trends.to_csv('working\\country_trends.csv', index=False)

report = {
    'total_patents': len(patents_clean),
    'top_inventors': top_inventors.to_dict(orient='records'),
    'top_companies': top_companies.to_dict(orient='records'),
    'top_countries': country_trends.head(10).to_dict(orient='records')
}

with open('working\\report.json','w') as f:
    json.dump(report, f, indent=4)

print('Reports generated')